In [1]:
import datetime as dt
import numpy as np
import tensorflow as tf

timestamp = dt.datetime.now().strftime('%Y%m%d%H%M%S')

xy = np.loadtxt('train.txt', unpack=True)
x_data = np.transpose(xy[0:-1])
y_data = np.reshape(xy[-1], (4,1))

X = tf.placeholder(tf.float32, name = 'X-input')
Y = tf.placeholder(tf.float32, name = 'Y-input')

W1 = tf.Variable(tf.random_uniform([2, 2], -1.0, 1.0), name = 'Weight1')
W2 = tf.Variable(tf.random_uniform([2, 1], -1.0, 1.0), name = 'Weight2')

b1 = tf.Variable(tf.zeros([2]), name = 'Bias1')
b2 = tf.Variable(tf.zeros([1]), name = 'Bias2')

w1_buf = tf.histogram_summary('Weights1', W1)
w2_buf = tf.histogram_summary('Weights2', W2)

b1_buf = tf.histogram_summary('Biases1', b1)
b2_buf = tf.histogram_summary('Biases2', b2)

In [2]:
with tf.name_scope('Layer2') as scope:
    L2 = tf.nn.relu(tf.matmul(X, W1) + b1)

with tf.name_scope('Layer3') as scope:
    hypothesis = tf.sigmoid(tf.matmul(L2, W2) + b2)

with tf.name_scope('Cost') as scope:
    cost = -tf.reduce_mean(Y * tf.log(hypothesis) + (1 - Y) * tf.log(1. - hypothesis))
    cost_buf = tf.scalar_summary('Cost', cost)

with tf.name_scope('Train') as scope:
    optimizer = tf.train.GradientDescentOptimizer(0.5)
    train = optimizer.minimize(cost)

with tf.name_scope('Accuracy') as scope:
    accuracy = tf.reduce_mean(tf.cast(tf.equal(tf.floor(hypothesis + 0.5), Y), tf.float32))
    accuracy_buf = tf.scalar_summary('Accuracy', accuracy)

In [3]:
with tf.Session() as sess:    
    writer = tf.train.SummaryWriter('/logs/nn_relu/train_' + timestamp, sess.graph)
    merged = tf.merge_all_summaries()
    
    tf.initialize_all_variables().run()
    
    for step in range(8000):
        sess.run(train, feed_dict = {X: x_data, Y: y_data})
        if step % 200 == 0:
            summary = sess.run(merged, feed_dict = {X: x_data, Y: y_data})
            writer.add_summary(summary, step)

    # Test model
    correct_prediction = tf.equal(tf.floor(hypothesis + 0.5), Y)
    print(sess.run(correct_prediction, feed_dict={X:x_data, Y:y_data}))

[[ True]
 [ True]
 [ True]
 [ True]]
